# Training SFT (QLoRA) — CoT vs non-CoT
Latih student `Qwen2.5-1.5B`/`0.5B` dari `data/sft/train/cot.jsonl` & `nocot.jsonl`.
Hyperparam identik, beda **cuma dataset** -> isolasi efek CoT.

Data train sudah jadi & ikut di repo (hasil `src.training.build_train`: teacher DeepSeek, merge numglue+un, decontam vs test). Cell §3 cuma **verifikasi** file ada — tak perlu Add Input data lagi.

**Setting Kaggle:** GPU **T4 x2** (1 GPU cukup utk 1.5B QLoRA), Internet **On**.

### 1. Install

In [ ]:
!pip install -q -U unsloth unsloth_zoo
!pip install -q -U datasets pyyaml

### 2. Clone/pull repo (permanen)
Repo private -> butuh PAT. Simpan Kaggle Secret `GH_TOKEN` (fine-grained, Contents:Read).
Kalau repo sudah public, hapus bagian token.

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = '/kaggle/working/FP_NLP'
TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
except Exception as e:
    print('no GH_TOKEN secret (ok kalau repo public):', e)
auth = f'{TOKEN}@' if TOKEN else ''
URL = f'https://{auth}github.com/henray404/FP_NLP.git'
if os.path.exists(REPO):
    # Konsumer kode (read-only): samain PERSIS ke remote (jangan 'pull' -> bisa divergen).
    subprocess.run(['git','-C',REPO,'fetch','-q',URL,'main'], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','FETCH_HEAD'], check=True)
else:
    subprocess.run(['git','clone','-q',URL,REPO], check=True)
sys.path.insert(0, REPO)
os.chdir(REPO)
print('repo ready (disamain ke origin/main):', REPO)

### 3. Cek dataset SFT (sudah di repo)
Data train final ada di `data/sft/train/cot.jsonl` + `nocot.jsonl` (1 set per arm, sudah merge numglue+un & decontam vs test). Cell ini cuma pastikan file ada + hitung baris. cot & nocot harus **sama jumlahnya** (problem identik -> perbandingan fair).

Mau bangun ulang dari sumber? jalankan `python -m src.training.build_train`.

In [ ]:
from pathlib import Path

def count_lines(p):
    return sum(1 for l in open(p, encoding='utf-8') if l.strip())

COT   = 'data/sft/train/cot.jsonl'
NOCOT = 'data/sft/train/nocot.jsonl'
assert Path(COT).exists() and Path(NOCOT).exists(), \
    'data train tak ada -- jalankan: python -m src.training.build_train'

n_cot, n_nocot = count_lines(COT), count_lines(NOCOT)
print(f'cot   : {n_cot} baris -> {COT}')
print(f'nocot : {n_nocot} baris -> {NOCOT}')
assert n_cot == n_nocot, f'cot ({n_cot}) != nocot ({n_nocot}) -- coverage tak fair!'
print(f'\nOK: {n_cot} problem identik di kedua arm (fair).')

### 4. Konfigurasi run
Pilih config (1.5B default). `MAX_EXAMPLES` kecil dulu buat smoke test, lalu `None` utk full.

In [ ]:
from src.training.train_sft import load_config, build_and_train
CONFIGS = ['src/training/configs/cot_1.5b.yaml',
           'src/training/configs/nocot_1.5b.yaml']
MAX_EXAMPLES = 50   # None = full
OUT_ROOT = '/kaggle/working'

### 5. Train (CoT lalu non-CoT)

In [ ]:
import gc, torch
for cfg_path in CONFIGS:
    cfg = load_config(cfg_path, max_examples=MAX_EXAMPLES)
    cfg.output_dir = f"{OUT_ROOT}/adapter_{Path(cfg_path).stem}"
    print('==='*12, cfg.mode, cfg.base_model, '->', cfg.output_dir)
    build_and_train(cfg)
    gc.collect(); torch.cuda.empty_cache()

### 6. Zip adapter -> Output
Eval pakai `notebooks/skenario4_eval_kaggle.ipynb` (load base + adapter sebagai 'model kita').

In [ ]:
!cd /kaggle/working && zip -rq adapters_sft.zip adapter_* && echo 'adapters_sft.zip siap di Output'